In [47]:
import pandas as pd
import requests
from Bio.PDB import PDBParser
import numpy as np
import matplotlib.pyplot as plt
import os


In [41]:
file_path = r'train_seq.txt'
import pandas as pd

df = pd.DataFrame(columns=['Entry', 'Sequence'])
# df = df.loc[len(df)] = ['A', 'B']

with open(file_path, 'r', encoding='utf-8') as f:
  while True:
    entry = f.readline().strip()
    if not entry:
      break
    sequence = f.readline().strip()
    df.loc[len(df)] = [entry[1:], sequence]

In [42]:
df

,Entry,Sequence
0,Q5I0E9,MEVLEEPAPGPGGADAAERRGLRRLLLSGFQEELRALLVLAGPAFL...
1,P63033,MMKTLSSGNCTLNVPAKNSYRMVVLGASRVGKSSIVSRFLNGRFED...
2,Q9NR71,MAKRTFSNLETFLIFLLVMMSAITVALLSLLFITSGTIENHKDLGG...
3,Q86XT9,MGNCQAGHNLHLCLAHHPPLVCATLILLLLGLSGLGLGSFLLTHRT...
4,A2CI98,MDPSKQGTLNRVENSVYRTAFKLRSVQTLCQLDLMDSFLIQQVLWR...
...,...,...
10033,V5NC32,MFPRVVRLNSRLVSFALLGLQIANGAITYQHPDDLPSNVNYDFIVA...
10034,P86368,SLLEFGMMILEETGKLAVPFYSSYGCYCGWGGKATPKDATDRCCFV...
10035,P80156,TKCYKTGDRIISEACPPGQDLCYMKTWCDVFCGTRGRVIELGCTAT...
10036,Q8NIH1,MHGLLLAAAGLLSLPLHVVAHPQPSTSLAGRGVDLDAYRMADRSSY...


In [44]:
df['Entry'][0]

'Q5I0E9'

In [ ]:
def download_pdb(uniprot_id):
    """Download the PDB file for a given UniProt ID."""
    url = f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_v4.pdb"
    response = requests.get(url)
    
    if response.status_code == 200:
        pdb_filename = f"{uniprot_id}_structure.pdb"
        with open(pdb_filename, "wb") as file:
            file.write(response.content)
        print(f"Downloaded {pdb_filename}")
        return pdb_filename
    else:
        print(f"Error fetching data for {uniprot_id}: {response.status_code}")
        return None

In [54]:
def generate_contact_map(pdb_filename):
    """Generate and plot the contact map for a given PDB file."""
    # Parse the PDB structure
    parser = PDBParser()
    structure = parser.get_structure('protein', pdb_filename)
    
    # Get coordinates of C-alpha atoms
    coords = []
    for model in structure:
        for chain in model:
            for residue in chain:
                if residue.has_id('CA'):  # Use CA atoms for the contact map
                    coords.append(residue['CA'].get_coord())
    
    coords = np.array(coords)
    
    # Calculate the contact map
    threshold = 8.0  # Distance threshold in Å
    num_residues = len(coords)
    contact_map = np.zeros((num_residues, num_residues))

    for i in range(num_residues):
        for j in range(i + 1, num_residues):
            distance = np.linalg.norm(coords[i] - coords[j])
            if distance < threshold:
                contact_map[i, j] = 1
                contact_map[j, i] = 1 
    return contact_map
    

In [53]:
output_dir = 'graph_directory'  # Directory to save contact map files
os.makedirs(output_dir, exist_ok=True) 

In [56]:

for index, row in df.iterrows():
    uniprot_id = row['Entry']  # Get the UniProt ID
    print(f"Processing {uniprot_id}...")
    
    # Step 1: Download PDB file
    pdb_filename = download_pdb(uniprot_id)
    
    if pdb_filename:
        # Step 2: Generate the contact map matrix
        contact_map_matrix = generate_contact_map(pdb_filename)
        
        if contact_map_matrix is not None:
            # Step 3: Save the contact map as a NumPy .npy file using only the unique UniProt ID
            # Use the first part of the UniProt ID (before version number if any)
            uniprot_base_id = uniprot_id.split('-')[0]  # Take the part before any version suffix (e.g., P12345)
            output_file = os.path.join(output_dir, f"{uniprot_base_id}.npy")
            np.save(output_file, contact_map_matrix)  # Save the matrix as a .npy file
            print(f"Saved contact map for {uniprot_base_id} to {output_file}")
        
        # Step 4: Delete the PDB file after use to save space
        os.remove(pdb_filename)
        # print(f"Deleted {pdb_filename} to save space.")
    
    else:
        print(f"Skipping {uniprot_id} due to error in PDB download.")


Processing Q5I0E9...
Downloaded Q5I0E9_structure.pdb
Saved contact map for Q5I0E9 to graph_directory\Q5I0E9.npy
Processing P63033...
Downloaded P63033_structure.pdb
Saved contact map for P63033 to graph_directory\P63033.npy
Processing Q9NR71...
Downloaded Q9NR71_structure.pdb
Saved contact map for Q9NR71 to graph_directory\Q9NR71.npy
Processing Q86XT9...
Downloaded Q86XT9_structure.pdb
Saved contact map for Q86XT9 to graph_directory\Q86XT9.npy
Processing A2CI98...
Downloaded A2CI98_structure.pdb
Saved contact map for A2CI98 to graph_directory\A2CI98.npy
Processing O75326...
Downloaded O75326_structure.pdb
Saved contact map for O75326 to graph_directory\O75326.npy
Processing P93004...
Downloaded P93004_structure.pdb
Saved contact map for P93004 to graph_directory\P93004.npy
Processing Q96FE7...
Downloaded Q96FE7_structure.pdb
Saved contact map for Q96FE7 to graph_directory\Q96FE7.npy
Processing Q9BSF0...
Downloaded Q9BSF0_structure.pdb
Saved contact map for Q9BSF0 to graph_directory\Q9B

KeyboardInterrupt: 